In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import numpy as np
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from keras import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.models import * 
from tensorflow.keras.preprocessing import image

In [2]:
train_path= '../input/bone-fracture-detection-using-xrays/archive (6)/train'
test_path='../input/bone-fracture-detection-using-xrays/archive (6)/val'

In [3]:
train_datagen = image.ImageDataGenerator(
    rotation_range=15,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    width_shift_range=0.1,
    height_shift_range=0.1
)
val_datagen= image.ImageDataGenerator(    rotation_range=15,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    width_shift_range=0.1,
    height_shift_range=0.1)

In [4]:
train_generator = train_datagen.flow_from_directory(
    train_path,
    target_size = (224,224),
    batch_size = 4,
    class_mode = 'binary')
validation_generator = val_datagen.flow_from_directory(
    test_path,
    target_size = (224,224),
    batch_size = 4,
    shuffle=True,
    class_mode = 'binary')

Found 8863 images belonging to 2 classes.
Found 600 images belonging to 2 classes.


In [9]:
base_model = tf.keras.applications.EfficientNetB3(weights='imagenet', input_shape=(224,224,3), include_top=False)

for layer in base_model.layers:
    layer.trainable=False
model = Sequential()
model.add(base_model)
model.add(GaussianNoise(0.25))
model.add(GlobalAveragePooling2D())
model.add(Dense(512,activation='relu'))
model.add(BatchNormalization())
model.add(GaussianNoise(0.25))
model.add(Dropout(0.25))
model.add(Dense(1, activation='sigmoid'))
model.summary()

43950080/43941136 [==============================] - 0s 0us/step
Model: "sequential_1"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
efficientnetb3 (Functional)  (None, 7, 7, 1536)        10783535  
_________________________________________________________________
gaussian_noise_2 (GaussianNo (None, 7, 7, 1536)        0         
_________________________________________________________________
global_average_pooling2d_1 ( (None, 1536)              0         
_________________________________________________________________
dense_2 (Dense)              (None, 512)               786944    
_________________________________________________________________
batch_normalization_1 (Batch (None, 512)               2048      
_________________________________________________________________
gaussian_noise_3 (GaussianNo (None, 512)               0         
_______________________________________________________

In [10]:
model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy','Precision','Recall','AUC'])

In [11]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
lrp=ReduceLROnPlateau(monitor="val_loss", factor=0.1, patience=2)
filepath='best_model.h5'
checkpoint = ModelCheckpoint(filepath, monitor='val_accuracy', verbose=1, save_best_only=True, mode='max')
call=[checkpoint,lrp]
history = model.fit(
    train_generator,
    epochs=10,
    validation_data=validation_generator,
    steps_per_epoch= 50,
    callbacks=call
    )

Epoch 1/10
50/50 [==============================] - 22s 283ms/step - loss: 1.0230 - accuracy: 0.6100 - precision: 0.5743 - recall: 0.6237 - auc: 0.6747 - val_loss: 0.6594 - val_accuracy: 0.6450 - val_precision: 0.5354 - val_recall: 0.8500 - val_auc: 0.7502

Epoch 00001: val_accuracy improved from -inf to 0.64500, saving model to best_model.h5


/opt/conda/lib/python3.7/site-packages/keras/utils/generic_utils.py:497: CustomMaskWarning: Custom mask layers require a config and must override get_config. When loading, the custom mask layer must be passed to the custom_objects argument.
  category=CustomMaskWarning)


Epoch 2/10
50/50 [==============================] - 12s 251ms/step - loss: 0.9734 - accuracy: 0.5800 - precision: 0.6392 - recall: 0.5586 - auc: 0.6206 - val_loss: 0.9364 - val_accuracy: 0.5283 - val_precision: 0.4077 - val_recall: 0.3958 - val_auc: 0.5325

Epoch 00002: val_accuracy did not improve from 0.64500
Epoch 3/10
50/50 [==============================] - 12s 239ms/step - loss: 0.7662 - accuracy: 0.6400 - precision: 0.6105 - recall: 0.6237 - auc: 0.7118 - val_loss: 0.8705 - val_accuracy: 0.6317 - val_precision: 0.5225 - val_recall: 0.9208 - val_auc: 0.7300

Epoch 00003: val_accuracy did not improve from 0.64500
Epoch 4/10
50/50 [==============================] - 12s 238ms/step - loss: 0.6040 - accuracy: 0.6950 - precision: 0.6937 - recall: 0.7404 - auc: 0.7665 - val_loss: 0.7330 - val_accuracy: 0.6950 - val_precision: 0.5836 - val_recall: 0.8292 - val_auc: 0.7493

Epoch 00004: val_accuracy improved from 0.64500 to 0.69500, saving model to best_model.h5
Epoch 5/10
50/50 [========

In [12]:
model.evaluate(train_generator)

2216/2216 [==============================] - 147s 66ms/step - loss: 0.4834 - accuracy: 0.7731 - precision: 0.7547 - recall: 0.8017 - auc: 0.8584


[0.4833647906780243,
 0.7731016874313354,
 0.7547250986099243,
 0.8017339706420898,
 0.8584474921226501]

In [13]:
model.evaluate(validation_generator)

150/150 [==============================] - 10s 64ms/step - loss: 0.7951 - accuracy: 0.6333 - precision: 0.5431 - recall: 0.5250 - auc: 0.7041


[0.7950971126556396,
 0.6333333253860474,
 0.5431034564971924,
 0.5249999761581421,
 0.7041031122207642]

In [ ]:
from tensorflow.keras.utils import load_img, img_to_array
img = load_img('../input/bone-fracture-detection-using-xrays/archive (6)/val/not fractured/2-rotated1.jpg',target_size=(224,224))
imag = img_to_array(img)
imaga = np.expand_dims(imag,axis=0) 
ypred = model.predict(imaga)
print(ypred)
a=ypred[0]
if a<0.5:
      op="Fracture"   
else:
      op="Normal"
plt.imshow(img)
print("THE UPLOADED X-RAY IMAGE IS: "+str(op))